# 函数定义

## 二维 pin-jointed 2节点全局刚度矩阵

In [3]:
def stiffness(EA, L, theta, degree=False):
    '''
    2自由度pin-jointed二节点单元
    全局刚度矩阵
    '''
    if degree:
        theta = np.deg2rad(theta)
        
    c = np.cos(theta)
    s = np.sin(theta)
    cs = c*s
    c2, s2 = c**2, s**2

    k = EA / L * np.array([[c2, cs, -c2, -cs],
                              [cs, s2, -cs, -s2],
                              [-c2, -cs, c2, cs],
                              [-cs, -s2, cs, s2]])
    
    return k
    

## 组装刚度矩阵

In [4]:
def assemble_K(ke_list, connectivity, n_nodes, numbering="one"):
    """
    根据已经计算好的单元全局刚度矩阵组装总体刚度矩阵

    Parameters
    ----------
    ke_list : list
        每个单元的 4x4 全局刚度矩阵，例如 [ke1, ke2, ke3]

    connectivity : list
        每个单元连接的两个节点，例如：
        [(1, 2), (2, 3), (2, 4)]
        表示 element 1: node 1 - node 2
             element 2: node 2 - node 3
             element 3: node 2 - node 4

    n_nodes : int
        总节点数

    numbering : str
        "one"  表示节点编号从 1 开始
        "zero" 表示节点编号从 0 开始

    Returns
    -------
    K : ndarray
        总体刚度矩阵
    """

    dof_per_node = 2
    total_dof = n_nodes * dof_per_node

    K = np.zeros((total_dof, total_dof))

    for ke, conn in zip(ke_list, connectivity):
        ni, nj = conn

        if numbering == "one":
            ni -= 1
            nj -= 1

        dofs = [
            2 * ni,      # ui
            2 * ni + 1,  # vi
            2 * nj,      # uj
            2 * nj + 1   # vj
        ]

        K[np.ix_(dofs, dofs)] += ke

    return K

## 自由度编号函数

In [5]:
def dof(node, direction, numbering="one"):
    """
    返回二维节点自由度编号

    Parameters
    ----------
    node : int
        节点编号

    direction : str
        "u" 或 "x" 表示水平位移自由度
        "v" 或 "y" 表示竖直位移自由度

    numbering : str
        "one"  表示节点编号从 1 开始
        "zero" 表示节点编号从 0 开始
    """

    if numbering == "one":
        node = node - 1

    direction = direction.lower()

    if direction in ["u", "x"]:
        return 2 * node
    elif direction in ["v", "y"]:
        return 2 * node + 1
    else:
        raise ValueError("direction must be 'u', 'v', 'x', or 'y'")

## 求解函数

In [6]:
def solve_structure(K, force_bc=None, disp_bc=None):
    """
    求解总体刚度方程 K d = F

    Parameters
    ----------
    K : ndarray
        总体刚度矩阵

    force_bc : dict
        已知外力边界条件，格式为：
        {
            dof_index: force_value
        }

        例如：
        {
            3: -50
        }

    disp_bc : dict
        已知位移边界条件，格式为：
        {
            dof_index: displacement_value
        }

        例如：
        {
            0: 0,
            1: 0,
            4: 0,
            5: 0
        }

    Returns
    -------
    d : ndarray
        完整位移向量

    reaction : ndarray
        完整反力向量

    free_dofs : ndarray
        自由自由度编号

    fixed_dofs : ndarray
        约束自由度编号
    """

    K = np.asarray(K, dtype=float)
    n_dof = K.shape[0]

    if K.shape[0] != K.shape[1]:
        raise ValueError("K must be a square matrix.")

    # -------------------------
    # 1. 定义总外力向量 F
    # -------------------------
    F = np.zeros(n_dof)

    if force_bc is not None:
        for index, value in force_bc.items():
            F[index] = value

    # -------------------------
    # 2. 定义已知位移向量 d
    # -------------------------
    d = np.zeros(n_dof)

    if disp_bc is None:
        disp_bc = {}

    fixed_dofs = np.array(sorted(disp_bc.keys()), dtype=int)

    for index, value in disp_bc.items():
        d[index] = value

    # -------------------------
    # 3. 找自由自由度
    # -------------------------
    all_dofs = np.arange(n_dof)
    free_dofs = np.setdiff1d(all_dofs, fixed_dofs)

    # -------------------------
    # 4. 分块矩阵
    # -------------------------
    K_ff = K[np.ix_(free_dofs, free_dofs)]
    K_fc = K[np.ix_(free_dofs, fixed_dofs)]

    F_f = F[free_dofs]
    d_c = d[fixed_dofs]

    # -------------------------
    # 5. 求解自由位移
    # -------------------------
    rhs = F_f - K_fc @ d_c
    d_f = np.linalg.solve(K_ff, rhs)

    d[free_dofs] = d_f

    # -------------------------
    # 6. 计算支座反力
    # -------------------------
    internal_force = K @ d
    reaction = internal_force - F

    return d, reaction, free_dofs, fixed_dofs

In [ ]:
def bar_results_2d(d, connectivity, EA_list, L_list, theta_list,
                      A_list=None, degree=True, numbering="one", tol=1e-10):
    """
    使用 EA 作为整体变量，计算二维杆单元轴力、应变、伸缩量、受拉/受压。
    若提供 A_list，则同时计算应力 stress。
    """

    d = np.asarray(d, dtype=float)
    n_elem = len(connectivity)

    def expand(x, name):
        if x is None:
            return None
        if np.isscalar(x):
            return [x] * n_elem
        if len(x) != n_elem:
            raise ValueError(f"{name} 的长度必须等于单元数量")
        return list(x)

    EA_list = expand(EA_list, "EA_list")
    L_list = expand(L_list, "L_list")
    theta_list = expand(theta_list, "theta_list")
    A_list = expand(A_list, "A_list")

    results = []

    for e, conn in enumerate(connectivity):
        ni, nj = conn

        if numbering == "one":
            ni -= 1
            nj -= 1
        elif numbering != "zero":
            raise ValueError("numbering must be 'one' or 'zero'.")

        dofs = [2 * ni, 2 * ni + 1, 2 * nj, 2 * nj + 1]
        ui, vi, uj, vj = d[dofs]

        EA = EA_list[e]
        L = L_list[e]
        theta = theta_list[e]

        if degree:
            theta = np.deg2rad(theta)

        c = np.cos(theta)
        s = np.sin(theta)

        elongation = c * (uj - ui) + s * (vj - vi)
        strain = elongation / L
        force = EA * strain

        if A_list is None:
            stress = None
        else:
            stress = force / A_list[e]

        if force > tol:
            state = "受拉"
        elif force < -tol:
            state = "受压"
        else:
            state = "零力杆"

        results.append({
            "element": e + 1,
            "node_i": conn[0],
            "node_j": conn[1],
            "EA": EA,
            "elongation": elongation,
            "strain": strain,
            "stress": stress,
            "force": force,
            "state": state
        })

    return results
